# 03 — Fine-tune GraphCodeBERT (binary vulnerability classifier)

**Run this notebook on Google Colab with a GPU** (`Runtime` → `Change runtime type` →
`T4 GPU`). Fine-tuning ~132K functions on CPU would take many hours; on a free
Colab T4 it takes roughly 1–2 hours for 3 epochs.

It also runs locally (CPU) for quick code checks — it auto-detects the
environment and adjusts data paths accordingly.

## Before running on Colab
Upload these 4 files (created in Phase 3) to your Google Drive at
`MyDrive/vulndetect-cpp/data/processed/`:
- `train.parquet`
- `val.parquet`
- `test.parquet`
- `class_weights.json`

(Just drag-and-drop the `data/processed` folder into Google Drive —
create the `vulndetect-cpp` folder first so the path matches.)

In [ ]:
try:
    import google.colab  # noqa: F401
    IS_COLAB = True
except ImportError:
    IS_COLAB = False
print("Running on Colab:", IS_COLAB)

In [ ]:
if IS_COLAB:
    %pip install -q transformers accelerate scikit-learn pandas pyarrow tqdm

In [ ]:
from pathlib import Path

if IS_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    DATA_DIR = Path("/content/drive/MyDrive/vulndetect-cpp/data/processed")
    MODEL_OUT_DIR = Path("/content/drive/MyDrive/vulndetect-cpp/models/graphcodebert_finetuned")
else:
    DATA_DIR = Path("../data/processed")
    MODEL_OUT_DIR = Path("../models/graphcodebert_finetuned")

MODEL_OUT_DIR.mkdir(parents=True, exist_ok=True)
print("DATA_DIR:", DATA_DIR)
print("MODEL_OUT_DIR:", MODEL_OUT_DIR)

## Hyperparameters

- `MAX_LENGTH = 512` — GraphCodeBERT's hard limit (it's a RoBERTa-based model
  with 512 position embeddings). From `01_eda.ipynb`/token-length analysis,
  ~18% of functions in our dataset are longer than this and get truncated —
  a known limitation, not a bug.
- `SUBSET_SIZE` — set to a small number (e.g. `200`) for a **quick smoke test**
  (~2–3 min) to confirm the whole pipeline runs before committing to a full
  multi-hour run. Set to `None` for the real training run.
- `BATCH_SIZE = 16` fits comfortably on a T4 GPU (16GB) at `MAX_LENGTH=512`.
  Lower it (e.g. to 8) if you hit a CUDA out-of-memory error.

In [ ]:
MODEL_NAME = "microsoft/graphcodebert-base"
MAX_LENGTH = 512
BATCH_SIZE = 16
NUM_EPOCHS = 3
LEARNING_RATE = 2e-5
SUBSET_SIZE = None  # e.g. 200 for a quick smoke test; None = full dataset

## 1. Load data

In [ ]:
import json

import pandas as pd

train_df = pd.read_parquet(DATA_DIR / "train.parquet")
val_df = pd.read_parquet(DATA_DIR / "val.parquet")

with open(DATA_DIR / "class_weights.json") as f:
    class_weights = json.load(f)

if SUBSET_SIZE is not None:
    train_df = train_df.sample(min(SUBSET_SIZE, len(train_df)), random_state=42).reset_index(drop=True)
    val_df = val_df.sample(min(SUBSET_SIZE // 4, len(val_df)), random_state=42).reset_index(drop=True)

print(f"train: {len(train_df)}  val: {len(val_df)}")
print("class_weights:", class_weights)

## 2. Tokenizer + PyTorch Dataset

Each `func_before` string is tokenized into `input_ids` (numeric token IDs)
and `attention_mask` (marks real tokens vs. padding). We pad every sequence
to `MAX_LENGTH` for simplicity — slightly less compute-efficient than dynamic
padding, but much easier to reason about as a beginner.

In [ ]:
import torch
from torch.utils.data import Dataset
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)


class CodeDataset(Dataset):
    def __init__(self, df, tokenizer, max_length):
        self.texts = df["func_before"].tolist()
        self.labels = df["vul"].tolist()
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        enc = self.tokenizer(
            self.texts[idx],
            truncation=True,
            max_length=self.max_length,
            padding="max_length",
            return_tensors="pt",
        )
        return {
            "input_ids": enc["input_ids"].squeeze(0),
            "attention_mask": enc["attention_mask"].squeeze(0),
            "labels": torch.tensor(self.labels[idx], dtype=torch.long),
        }


train_dataset = CodeDataset(train_df, tokenizer, MAX_LENGTH)
val_dataset = CodeDataset(val_df, tokenizer, MAX_LENGTH)
print(f"train_dataset: {len(train_dataset)}  val_dataset: {len(val_dataset)}")

## 3. DataLoaders — weighted sampling for train

`WeightedRandomSampler` uses the per-class weights from Phase 3
(`class_weights.json`) so each training batch sees roughly balanced classes,
instead of being ~94% "not vulnerable" by default.

In [ ]:
from torch.utils.data import DataLoader, WeightedRandomSampler

sample_weights = train_df["vul"].map(lambda v: class_weights[str(v)]).to_numpy()
sampler = WeightedRandomSampler(
    weights=torch.as_tensor(sample_weights, dtype=torch.double),
    num_samples=len(sample_weights),
    replacement=True,
)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, sampler=sampler)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)
print(f"train batches: {len(train_loader)}  val batches: {len(val_loader)}")

## 4. Model, optimizer, scheduler

In [ ]:
from torch.optim import AdamW
from transformers import AutoModelForSequenceClassification, get_linear_schedule_with_warmup

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=2).to(device)

optimizer = AdamW(model.parameters(), lr=LEARNING_RATE)
total_steps = len(train_loader) * NUM_EPOCHS
scheduler = get_linear_schedule_with_warmup(
    optimizer, num_warmup_steps=int(0.1 * total_steps), num_training_steps=total_steps
)
print("device:", device)

## 5. Evaluation function

**Why not just accuracy?** With ~17:1 imbalance, a model that always predicts
"not vulnerable" scores ~94% accuracy while being useless. We track
precision/recall/F1 **for the vulnerable class specifically**, plus ROC-AUC.

In [ ]:
import torch.nn.functional as F
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, roc_auc_score


@torch.no_grad()
def evaluate(loader):
    model.eval()
    all_preds, all_labels, all_probs = [], [], []
    for batch in loader:
        batch = {k: v.to(device) for k, v in batch.items()}
        outputs = model(**batch)
        probs = F.softmax(outputs.logits, dim=-1)[:, 1]
        preds = outputs.logits.argmax(dim=-1)
        all_preds.extend(preds.cpu().tolist())
        all_labels.extend(batch["labels"].cpu().tolist())
        all_probs.extend(probs.cpu().tolist())

    precision, recall, f1, _ = precision_recall_fscore_support(
        all_labels, all_preds, average="binary", pos_label=1, zero_division=0
    )
    acc = accuracy_score(all_labels, all_preds)
    try:
        auc = roc_auc_score(all_labels, all_probs)
    except ValueError:
        auc = float("nan")  # only one class present (can happen on tiny smoke-test subsets)
    return {"accuracy": acc, "precision": precision, "recall": recall, "f1": f1, "roc_auc": auc}

## 6. Training loop

Saves the model to `MODEL_OUT_DIR` only when validation F1 (vulnerable class)
improves — this is the checkpoint Phase 6 will export to ONNX.

In [ ]:
from tqdm.auto import tqdm

best_f1 = -1.0
history = []

for epoch in range(NUM_EPOCHS):
    model.train()
    running_loss = 0.0
    pbar = tqdm(train_loader, desc=f"Epoch {epoch + 1}/{NUM_EPOCHS}")
    for step, batch in enumerate(pbar, start=1):
        batch = {k: v.to(device) for k, v in batch.items()}
        optimizer.zero_grad()
        outputs = model(**batch)
        loss = outputs.loss
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        scheduler.step()
        running_loss += loss.item()
        pbar.set_postfix(loss=running_loss / step)

    metrics = evaluate(val_loader)
    metrics["epoch"] = epoch + 1
    metrics["train_loss"] = running_loss / len(train_loader)
    history.append(metrics)
    print(f"Epoch {epoch + 1}: {metrics}")

    if metrics["f1"] > best_f1:
        best_f1 = metrics["f1"]
        model.save_pretrained(MODEL_OUT_DIR)
        tokenizer.save_pretrained(MODEL_OUT_DIR)
        print(f"  -> New best F1={best_f1:.4f}, saved to {MODEL_OUT_DIR}")

print("\nTraining history:")
pd.DataFrame(history)

## 7. Getting the model back to your local machine

If `MODEL_OUT_DIR` was on Google Drive (`IS_COLAB=True`), the saved model is
already in your Drive at `MyDrive/vulndetect-cpp/models/graphcodebert_finetuned/`
— just download that folder from the Drive web UI (right-click → Download)
into your local `models/graphcodebert_finetuned/` so Phase 6 (ONNX export)
can find it.